# CRM AI project


The AI converts that request into SQL, your Python program executes the query, and the result is returned to you.


1. User asks a question
```
"Show all customers from Kerala"
```
2. Python reads database metadata
```
Tables, columns, data types and foreign-key relationships
```

3. Gemini generates SQL
```
The question and schema are sent to the AI model
```

4. Python validates the SQL
```
The notebook blocks queries containing DROP or TRUNCATE
```

5. MySQL executes the query
```
SELECT retrieves data; other statements may change data
```

6. Python returns the result
```
A DataFrame, success message, or AI-generated error explanation
```

In [2]:
!pip install pymysql
!pip install plotly
!pip install nbformat


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install google-genai pandas


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install python-dotenv


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
from google import genai
import pymysql
import pandas as pd
import plotly.express as px

In [6]:
# Establish connection using PyMySQL
conn = pymysql.connect(
    host="127.0.0.1",  # Using IP explicitly avoids localhost/IPv6 name resolution quirks
    port=3306,
    user="root",
    password="admin",
    database="crm_ai"
)

cursor = conn.cursor()
print("Connected successfully!")

Connected successfully!


## Schema Definition & Gemini Model Setup

### Initial manual text representation of the database schema

In [7]:
SCHEMA = """
DATABASE : crm_ai

TABLE: customers

customer_id INT PRIMARY KEY AUTO_INCREMENT
name VARCHAR(100)
email VARCHAR(100)
phone VARCHAR(20)
city VARCHAR(50)
state VARCHAR(50)
created_at DATETIME

TABLE: products

product_id INT PRIMARY KEY AUTO_INCREMENT
product_name VARCHAR(100)
category VARCHAR(50)
price DECIMAL(10,2)

TABLE: orders

order_id INT PRIMARY KEY AUTO_INCREMENT
customer_id INT FOREIGN KEY REFERENCES customers(customer_id)
order_date DATE
total_amount DECIMAL(10,2)
status VARCHAR(20)

TABLE: order_items

item_id INT PRIMARY KEY AUTO_INCREMENT
order_id INT FOREIGN KEY REFERENCES orders(order_id)
product_id INT FOREIGN KEY REFERENCES products(product_id)
quantity INT
amount DECIMAL(10,2)

TABLE: leads

lead_id INT PRIMARY KEY AUTO_INCREMENT
lead_name VARCHAR(100)
source VARCHAR(50)
status VARCHAR(30)
assigned_to VARCHAR(50)

TABLE: employees

employee_id INT PRIMARY KEY AUTO_INCREMENT
employee_name VARCHAR(100)
department VARCHAR(50)
email VARCHAR(100)

RELATIONSHIPS:

customers.customer_id
       |
       |
orders.customer_id

orders.order_id
       |
       |
order_items.order_id

products.product_id
       |
       |
order_items.product_id
"""

### SDK client and lists all available Gemini models

In [8]:
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
print(os.getenv("GEMINI_API_KEY"))

<REDACTED>


In [10]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# List available models
for model in client.models.list():
    print(f"Name: {model.name}")
    print(f"Display Name: {model.display_name}")
    print("-" * 40)

Name: models/gemini-2.5-flash
Display Name: Gemini 2.5 Flash
----------------------------------------
Name: models/gemini-2.5-pro
Display Name: Gemini 2.5 Pro
----------------------------------------
Name: models/gemini-2.5-flash-preview-tts
Display Name: Gemini 2.5 Flash Preview TTS
----------------------------------------
Name: models/gemini-2.5-pro-preview-tts
Display Name: Gemini 2.5 Pro Preview TTS
----------------------------------------
Name: models/gemma-4-26b-a4b-it
Display Name: Gemma 4 26B A4B IT
----------------------------------------
Name: models/gemma-4-31b-it
Display Name: Gemma 4 31B IT
----------------------------------------
Name: models/gemini-flash-latest
Display Name: Gemini Flash Latest
----------------------------------------
Name: models/gemini-flash-lite-latest
Display Name: Gemini Flash-Lite Latest
----------------------------------------
Name: models/gemini-pro-latest
Display Name: Gemini Pro Latest
----------------------------------------
Name: models/gemin

### Initialize Gemini Client

In [11]:
# The client automatically picks up GEMINI_API_KEY from environment variables
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# Generate content directly using the client
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Hello, Gemini!"
)

print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

### Dynamic Database Metadata Extraction

In [ ]:
cursor.execute("""
SELECT
table_name,
column_name,
data_type
FROM information_schema.columns
WHERE table_schema='crm_ai'
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

('customers', 'city', 'varchar')
('customers', 'created_at', 'datetime')
('customers', 'customer_id', 'int')
('customers', 'email', 'varchar')
('customers', 'name', 'varchar')
('customers', 'phone', 'varchar')
('customers', 'state', 'varchar')
('employees', 'department', 'varchar')
('employees', 'email', 'varchar')
('employees', 'employee_id', 'int')
('employees', 'employee_name', 'varchar')
('leads', 'assigned_to', 'varchar')
('leads', 'lead_id', 'int')
('leads', 'lead_name', 'varchar')
('leads', 'source', 'varchar')
('leads', 'status', 'varchar')
('order_items', 'amount', 'decimal')
('order_items', 'item_id', 'int')
('order_items', 'order_id', 'int')
('order_items', 'product_id', 'int')
('order_items', 'quantity', 'int')
('orders', 'customer_id', 'int')
('orders', 'order_date', 'date')
('orders', 'order_id', 'int')
('orders', 'status', 'varchar')
('orders', 'total_amount', 'decimal')
('products', 'category', 'varchar')
('products', 'price', 'decimal')
('products', 'product_id', 'int'

In [ ]:
cursor.execute("""
SELECT
table_name,
column_name
FROM information_schema.key_column_usage
WHERE table_schema='crm_ai'
AND constraint_name='PRIMARY'
""")

pk_rows = cursor.fetchall()

print(pk_rows)

(('customers', 'customer_id'), ('employees', 'employee_id'), ('leads', 'lead_id'), ('order_items', 'item_id'), ('orders', 'order_id'), ('products', 'product_id'))


In [ ]:
def get_schema():

    query = """
    SELECT
        table_name,
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_schema='crm_ai'
    ORDER BY table_name, ordinal_position
    """

    cursor.execute(query)

    rows = cursor.fetchall()

    schema = ""

    current_table = None

    for table, column, dtype, nullable in rows:

        if current_table != table:

            schema += f"\n\nTABLE: {table}\n"

            current_table = table

        schema += (
            f"   {column} "
            f"({dtype}) "
            f"NULLABLE={nullable}\n"
        )

    return schema

In [ ]:
def get_relationships():

    query = """
    SELECT
        table_name,
        column_name,
        referenced_table_name,
        referenced_column_name
    FROM information_schema.key_column_usage
    WHERE table_schema='crm_ai'
    AND referenced_table_name IS NOT NULL
    """

    cursor.execute(query)

    rows = cursor.fetchall()

    relationships = "\n\nRELATIONSHIPS:\n"

    for row in rows:

        relationships += (
            f"{row[0]}.{row[1]}"
            f" --> "
            f"{row[2]}.{row[3]}\n"
        )

    return relationships

In [ ]:
def build_context():

    schema = get_schema()

    relationships = get_relationships()

    context = schema + relationships

    return context

In [ ]:
build_context()

'\n\nTABLE: customers\n   customer_id (int) NULLABLE=NO\n   name (varchar) NULLABLE=YES\n   email (varchar) NULLABLE=YES\n   phone (varchar) NULLABLE=YES\n   city (varchar) NULLABLE=YES\n   state (varchar) NULLABLE=YES\n   created_at (datetime) NULLABLE=YES\n\n\nTABLE: employees\n   employee_id (int) NULLABLE=NO\n   employee_name (varchar) NULLABLE=YES\n   department (varchar) NULLABLE=YES\n   email (varchar) NULLABLE=YES\n\n\nTABLE: leads\n   lead_id (int) NULLABLE=NO\n   lead_name (varchar) NULLABLE=YES\n   source (varchar) NULLABLE=YES\n   status (varchar) NULLABLE=YES\n   assigned_to (varchar) NULLABLE=YES\n\n\nTABLE: order_items\n   item_id (int) NULLABLE=NO\n   order_id (int) NULLABLE=YES\n   product_id (int) NULLABLE=YES\n   quantity (int) NULLABLE=YES\n   amount (decimal) NULLABLE=YES\n\n\nTABLE: orders\n   order_id (int) NULLABLE=NO\n   customer_id (int) NULLABLE=YES\n   order_date (date) NULLABLE=YES\n   total_amount (decimal) NULLABLE=YES\n   status (varchar) NULLABLE=YES\n\

In [ ]:
def generate_sql(user_question):
    schema_context = build_context()
    prompt = f"""
    You are an expert MySQL developer.
    DATABASE SCHEMA:
    {schema_context}
    RULES:
    1. Return ONLY executable MySQL.
    2. No markdown, no explanations, no comments.
    3. Use joins when required.
    4. Never generate DROP or TRUNCATE.
    USER QUESTION:
    {user_question}
    """
    response = client.models.generate_content(model="gemini-3.8-flash", contents=prompt)
    sql = response.text.replace("```sql", "").replace("```", "").strip()
    return sql

In [ ]:
def validate_sql(sql):
    blocked_keywords = ["drop", "truncate"]
    sql_lower = sql.lower()
    for keyword in blocked_keywords:
        if keyword in sql_lower:
            raise Exception(f"Blocked Query Detected: {keyword}")
    return True

In [ ]:
import pandas as pd

In [ ]:
def explain_error(user_query, sql, error):
    prompt = f"""
    You are a CRM assistant.
    User Request: {user_query}
    Generated SQL: {sql}
    Database Error: {error}
    Explain the issue in simple business language and suggest what to do next.
    Do not show technical database jargon.
    """

    # Correct SDK call
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )
    return response.text

In [ ]:
def execute_sql_with_retry(sql, user_query, max_retries=2):
    """Executes SQL with an agentic self-correction loop if errors occur."""
    current_sql = sql
    
    for attempt in range(max_retries + 1):
        try:
            validate_sql(current_sql)
            cursor.execute(current_sql)
            
            if current_sql.lower().startswith("select"):
                rows = cursor.fetchall()
                columns = [d[0] for d in cursor.description]
                return pd.DataFrame(rows, columns=columns)
            
            conn.commit()
            return "Operation Successful"
            
        except Exception as e:
            error_msg = str(e)
            
            # If we've exhausted retries, fallback to friendly explanation
            if attempt == max_retries:
                return explain_error(user_query, current_sql, error_msg)
            
            # Self-Correction prompt to Gemini
            correction_prompt = f"""
            You are an expert MySQL developer fixing a broken query.
            
            Original User Request: {user_query}
            Failed SQL Query: {current_sql}
            MySQL Error: {error_msg}
            
            Please analyze the error, fix the SQL query, and return ONLY executable MySQL code without markdown or explanations.
            """
            
            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=correction_prompt
            )
            
            current_sql = (
                response.text
                .replace("```sql", "")
                .replace("```", "")
                .strip()
            )
            print(f"Retrying with corrected SQL (Attempt {attempt + 1}):\n{current_sql}")

In [ ]:
def classify_intent(user_query):
    """Classifies user input to ensure it falls within CRM boundaries."""
    prompt = f"""
    You are a security and intent router for a CRM AI assistant. 
    The database contains customers, products, orders, order items, leads, and employees.
    
    Analyze the user query: "{user_query}"
    
    Is this query related to querying, updating, or managing CRM data, business metrics, or entities?
    Answer ONLY with 'YES' or 'NO'.
    """
    
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )
    
    return response.text.strip().upper()

In [ ]:
def auto_visualize(df, user_query):
    """Automatically detects data patterns and renders a Plotly chart if appropriate."""
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return
    
    # Check if DataFrame has at least one categorical/date column and one numeric column
    cat_cols = df.select_dtypes(include=['object', 'str', 'category', 'datetime']).columns
    num_cols = df.select_dtypes(include=['number']).columns
    
    if len(cat_cols) >= 1 and len(num_cols) >= 1:
        x_col = cat_cols[0]
        y_col = num_cols[0]
        
        # Avoid plotting IDs as metrics
        if 'id' not in x_col.lower() and len(df) > 1:
            print(f"\nRendering interactive chart for: {user_query}")
            
            # Use bar chart for categories, line chart if it looks like time-series
            if 'date' in x_col.lower() or 'created' in x_col.lower():
                fig = px.line(df, x=x_col, y=y_col, markers=True, title=f"Trend: {user_query}")
            else:
                fig = px.bar(df, x=x_col, y=y_col, text_auto=True, title=f"Insights: {user_query}")
                
            fig.update_layout(template="plotly_dark", xaxis_tickangle=-45)
            fig.show()

In [ ]:
def ask_crm_secure(question):
    """Enhanced wrapper with intent checking and self-healing execution."""
    intent = classify_intent(question)
    
    if "NO" in intent:
        return "I'm your CRM assistant. I can only help you query, manage, or analyze customer, product, order, and lead data."
    
    sql = generate_sql(question)
    print("Generated SQL:\n", sql)
    
    result = execute_sql_with_retry(sql, question)
    return result

### Test Execution

In [ ]:
query = "Show all customers from Kerala"
result = ask_crm_secure(query)

display(result)


Generated SQL:
 SELECT * FROM customers WHERE state = 'Kerala';


,customer_id,name,email,phone,city,state,created_at
0,1,Arun,arun@gmail.com,9876543210,Kochi,Kerala,2026-10-22 09:39:34
1,2,Rahul,rahul@gmail.com,9876543211,Kochi,Kerala,2026-10-22 09:39:34
2,3,Anu,anu@gmail.com,9876543212,Thrissur,Kerala,2026-10-22 09:39:34
3,4,Priya,priya@gmail.com,9876543213,Calicut,Kerala,2026-10-22 09:39:34
4,10,Vishnu,vishnu@gmail.com,9876543219,Kochi,Kerala,2026-10-22 09:39:34
5,11,Akhil,akhil@gmail.com,9876543220,Kollam,Kerala,2026-10-22 09:39:34
6,12,Meera,meera@gmail.com,9876543221,Kannur,Kerala,2026-10-22 09:39:34
7,13,Nikhil,nikhil@gmail.com,9876543222,Kochi,Kerala,2026-10-22 09:39:34
8,15,Hari,hari@gmail.com,9876543224,Palakkad,Kerala,2026-10-22 09:39:34
9,17,Kiran,kiran@gmail.com,9876543226,Kochi,Kerala,2026-10-22 09:39:34


In [ ]:
auto_visualize(result, query)


Rendering interactive chart for: Show all customers from Kerala


In [ ]:
result = ask_crm_secure(
    "Show top 5 customers by revenue"
)
display(result)

Generated SQL:
 SELECT 
    c.customer_id, 
    c.name, 
    SUM(o.total_amount) AS total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_revenue DESC
LIMIT 5;


,customer_id,name,total_revenue
0,2,Rahul,95000.00
1,1,Arun,75000.00
2,20,Anjali,45000.00
3,16,Neha,35000.00
4,12,Meera,30000.00


In [ ]:
auto_visualize(result, "Show top 5 customers by revenue")


Rendering interactive chart for: Show top 5 customers by revenue


In [ ]:
result = ask_crm_secure(
    "List customers who purchased Laptop Pro"
)
display(result)

In [ ]:
ask_crm_secure(
    "Show pending orders"
)

In [ ]:
result = ask_crm_secure(
    "Delete all lost leads"
)
display(result)

Generated SQL:
 DELETE FROM leads WHERE status = 'Lost';


'Operation Successful'

In [ ]:
ask_crm_secure(
    "add one row"
)

Generated SQL:
 INSERT INTO customers (customer_id, name, email, phone, city, state, created_at) 
VALUES (1, 'John Doe', 'john.doe@example.com', '1234567890', 'New York', 'NY', NOW());
Retrying with corrected SQL (Attempt 1):
INSERT INTO customers (name, email, phone, city, state, created_at) 
VALUES ('John Doe', 'john.doe@example.com', '1234567890', 'New York', 'NY', NOW());


'Operation Successful'